In [ ]:
# Install build tools and git (Colab usually has most, but just in case)
!apt update -qq && apt install -y build-essential git cmake ninja-build -qq

# Clone llama.cpp (use --depth 1 to save time & space)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp

# Install Python dependencies needed by convert_hf_to_gguf.py
!pip install -r requirements.txt -q

In [5]:
# Build with CUDA support (recommended if you have GPU runtime)
!make clean
!make LLAMA_CUDA=1 -j$(nproc)

Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


In [3]:
import os
from huggingface_hub import snapshot_download
from pathlib import Path
import subprocess

# ────────────────────────────────────────────────
#          ←←←   CHANGE THESE THREE LINES   →→→
# ────────────────────────────────────────────────
MODEL_ID       = "gabrielbuzzi/medgemma-1.5-4b-it-text-only"   # ← your model
QUANT_METHOD   = "Q4_K_M"                             # Q4_K_M, Q5_K_M, Q6_K, Q8_0, etc.
HF_TOKEN       = ""                                   # only needed for gated models
# ────────────────────────────────────────────────

output_dir = Path("/content/gguf_output")
output_dir.mkdir(exist_ok=True, parents=True)

# 1. Download model from Hugging Face (safetensors preferred)
print(f"Downloading {MODEL_ID} ...")
model_path = snapshot_download(
    repo_id=MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    local_dir=output_dir / "hf_model",
    ignore_patterns=["*.msgpack", "*.h5"],   # prefer safetensors
)
print("Model downloaded to:", model_path)

# 2. Convert to f16 GGUF
model_name = MODEL_ID.split("/")[-1].replace(".", "-")   # clean name
f16_path = output_dir / f"{model_name}-F16.gguf"

print("\nConverting to f16 GGUF...")
subprocess.run([
    "python", "convert_hf_to_gguf.py",
    str(model_path),
    "--outfile", str(f16_path),
    "--outtype", "f16"
], check=True)

print(f"f16 GGUF created: {f16_path}")

# 3. Quantize to desired format
quant_path = output_dir / f"{model_name}-{QUANT_METHOD}.gguf"

print(f"\nQuantizing to {QUANT_METHOD} ...")
subprocess.run([
    "./llama-quantize",           # or ./build/bin/llama-quantize if built differently
    str(f16_path),
    str(quant_path),
    QUANT_METHOD
], check=True)

print(f"Done! Quantized model: {quant_path}")

# Optional: show file size
!ls -lh {output_dir}/*.gguf

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/2.13G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/2.13G [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/742 [00:00<?, ?B/s]

Model downloaded to: /content/gguf_output/hf_model

Converting to f16 GGUF...


CalledProcessError: Command '['python', 'convert_hf_to_gguf.py', '/content/gguf_output/hf_model', '--outfile', '/content/gguf_output/medgemma-1-5-4b-it-text-only-F16.gguf', '--outtype', 'f16']' returned non-zero exit status 1.